In [64]:
%load_ext autoreload
%autoreload 2

from utils_gae import *
from models_encoders import *

import numpy as np
import pandas as pd
import io
import html
import networkx as nx
import os
import sys

path_to_src = os.path.abspath(os.path.join('..'))
if path_to_src not in sys.path:
    sys.path.append(path_to_src)
from SHAP_like_graph_tool import load_all_data_for_graph, loadsave_data_joblib, hide_graph_links

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
def load_graphml_safe(path):
        with open(path, 'r', encoding='utf-8') as f:
            raw_data = f.read()

        clean_data = html.unescape(raw_data)
        G = nx.read_graphml(io.StringIO(clean_data))
        
        print(f"✅ Graphe chargé : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
        return G

In [61]:
for i in np.arange(0.00, 0.10, 0.10):
    sbm_val = f"{i:.2f}"
    pos_val = f"{1-i:.2f}"
    G_name = f"artificial_graph_sbmv4_{sbm_val.replace('.', '_')}_pos_{pos_val.replace('.', '_')}"
    G_name_obj = f"artificial_graph_sbmv4_{sbm_val.replace('.', '_')}_pos_{pos_val.replace('.', '_')}"

    G_kept = loadsave_data_joblib(filename=f"G_kept_w_struct_com_dist_{G_name}", mode="load")
    G_train, G_test = hide_graph_links(G_kept, test_size=0.15)
    
    # 1. Chargement des données d'entraînement
    G_train, dataset_train, dataset_eval, _, _, _ = load_all_data_for_graph(G_name)

    print(f"--- Inspection de {G_name} ---")
    
    # Affiche les attributs du premier nœud pour trouver la clé de position (ex: 'pos', 'coords', 'x'...)
    if len(G_train.nodes) > 0:
        first_node = list(G_train.nodes(data=True))[0]
        print(f"Attributs du premier nœud ({first_node[0]}) :", first_node[1])
    print("-" * 30)
    # ---------------------------
    
    kakak = prepare_graph_data(G_train, None, "GT_pos")
    kakak_eval = prepare_graph_data(G_test, None, "GT_pos")
    

Graphe original: 2501 liens
Graphe d'entraînement: 2125 liens
Liens cachés pour le test: 376
SHAP Analysis introuvable pour artificial_graph_sbmv4_0_00_pos_1_00.
--- Inspection de artificial_graph_sbmv4_0_00_pos_1_00 ---
Attributs du premier nœud (0) : {'GT_pos': array([0.92292958, 0.70103991, 0.73655641, 0.17676583]), 'degree': 20, 'pr': 0.004603413216892907, 'ppr': 0.004755349116032537, 'lcc': 0.4789473684210526, 'and': 25.1, 'dc': 0.10152284263959391, 'katz': 0.05204739626286779, 'n2v_homophily': array([ 0.13497628, -0.19414312, -0.10347388,  0.29276982,  0.09353141,
        0.11916769,  0.01776646,  0.20392649, -0.5796253 ,  0.31983784,
       -0.17546275, -0.07357487, -0.00607024,  0.0332728 , -0.11660729,
        0.52339244, -0.21624179,  0.2931292 ,  0.08369698,  0.06780953,
       -0.1689504 , -0.2744288 ,  0.5518766 , -0.23446208,  0.22479519,
       -0.18776613, -0.09331908, -0.22530574, -0.04908746,  0.02411713,
        0.4948722 ,  0.19274905,  0.05638051,  0.20896259, -0.4

In [65]:
print(kakak)
print(kakak_eval)

gae = GAE(in_channels=1, out_channels=16)
gae_disentangled = GAE(in_channels=1, out_channels=16)
geo = GeoEncoder(in_pos_dim=4, out_channels=16, scale=1.0)
geo_disentangled = GeoEncoder(in_pos_dim=4, out_channels=16, scale=1.0)

Joint = JointLinkPredictionModel(gae, geo)
Joint_disentangled = JointLinkPredictionModel(gae_disentangled, geo_disentangled)

Joint_disentangled.fit(kakak, epochs=201, lambda_ortho=0.1)
Joint.fit(kakak, epochs=201, disentangle=False)


def check_collapse(z_a, z_b):
    cos_sim = F.cosine_similarity(z_a, z_b).mean()
    print(f"Similarité Cosine moyenne entre espaces : {cos_sim.item():.4f}")

print("------DISENTANGLED----------")
z_a = Joint_disentangled.encoder_a(kakak.x, kakak.edge_index)
z_b = Joint_disentangled.encoder_b(kakak.pos)
check_collapse(z_a, z_b)
auc, ap = Joint_disentangled.evaluate(kakak_eval)
print(f"Scores sur eval set : {auc, ap}")
print("------CLASSIC----------")
z_a = Joint.encoder_a(kakak.x, kakak.edge_index)
z_b = Joint.encoder_b(kakak.pos)
check_collapse(z_a, z_b)
auc, ap = Joint.evaluate(kakak_eval)
print(f"Scores sur eval set : {auc, ap}")

Data(x=[198, 1], edge_index=[2, 4250], pos=[198, 4], num_nodes=198)
Data(x=[198, 1], edge_index=[2, 752], pos=[198, 4], num_nodes=198)
[DISENTANGLED] Ep 000 | Loss: 19.7456 | Rec: 0.7289 | Lambda*Ortho: 19.0166
Détails modèles | A: 1.220 | B: 0.725
[DISENTANGLED] Ep 010 | Loss: 2.6010 | Rec: 0.3633 | Lambda*Ortho: 2.2377
Détails modèles | A: 8.529 | B: 0.676
[DISENTANGLED] Ep 020 | Loss: 1.0407 | Rec: 0.3560 | Lambda*Ortho: 0.6847
Détails modèles | A: 4.018 | B: 0.688
[DISENTANGLED] Ep 030 | Loss: 0.9387 | Rec: 0.3031 | Lambda*Ortho: 0.6356
Détails modèles | A: 4.402 | B: 0.712
[DISENTANGLED] Ep 040 | Loss: 0.7150 | Rec: 0.2793 | Lambda*Ortho: 0.4357
Détails modèles | A: 3.033 | B: 0.729
[DISENTANGLED] Ep 050 | Loss: 0.5321 | Rec: 0.2705 | Lambda*Ortho: 0.2616
Détails modèles | A: 2.718 | B: 0.750
[DISENTANGLED] Ep 060 | Loss: 0.6342 | Rec: 0.2648 | Lambda*Ortho: 0.3694
Détails modèles | A: 2.454 | B: 0.759
[DISENTANGLED] Ep 070 | Loss: 0.8815 | Rec: 0.2558 | Lambda*Ortho: 0.6257
Détai